# mesh_to_flexible_dual_grid old/ref/new JIT comparison

This notebook JIT-compiles CPU, old, ref, and current full-pipeline implementations, runs original and subdivided `notebooks/test.glb`, canonicalizes outputs by voxel key, checks correctness, and compares CUDA cold/warm runtime plus PyTorch-visible peak memory.

In [1]:
from pathlib import Path
import os
import tempfile
import time
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch
import trimesh
from torch.utils.cpp_extension import load

ROOT = Path('/mnt/nvmefs/Projects/o-voxel-gpu')
CONDA_PREFIX = Path(os.environ.get('CONDA_PREFIX', '/home/quanta/.conda/envs/symm-enforce'))
os.environ['PATH'] = f'{CONDA_PREFIX / "bin"}:{os.environ.get("PATH", "")}'
os.environ['CC'] = str(CONDA_PREFIX / 'bin/gcc')
os.environ['CXX'] = str(CONDA_PREFIX / 'bin/g++')
os.environ['CUDAHOSTCXX'] = str(CONDA_PREFIX / 'bin/gcc')
os.environ['MAX_JOBS'] = str(os.cpu_count() or 1)

DEVICE = torch.device('cuda')
assert torch.cuda.is_available(), 'CUDA is required for this notebook'

GRID_SIZE = 512
CHUNK_TRIANGLES = 20000
BOUNDARY_CHUNK_STEPS = 64
SUBDIVIDE_AREA_THRESHOLD = 1.0 / (GRID_SIZE * GRID_SIZE)
SUBDIVIDE_MAX_ITERS = 12
WARM_REPEATS = 5

FACE_WEIGHT = 1.0
BOUNDARY_WEIGHT = 1.0
REGULARIZATION_WEIGHT = 1e-3

print('torch:', torch.__version__, 'cuda:', torch.version.cuda)
print('device:', torch.cuda.get_device_name(DEVICE))
print('MAX_JOBS:', os.environ['MAX_JOBS'])
print('GRID_SIZE:', GRID_SIZE, 'CHUNK_TRIANGLES:', CHUNK_TRIANGLES, 'BOUNDARY_CHUNK_STEPS:', BOUNDARY_CHUNK_STEPS)
print('weights:', FACE_WEIGHT, BOUNDARY_WEIGHT, REGULARIZATION_WEIGHT)
print('SUBDIVIDE_AREA_THRESHOLD:', SUBDIVIDE_AREA_THRESHOLD, 'SUBDIVIDE_MAX_ITERS:', SUBDIVIDE_MAX_ITERS)

torch: 2.6.0+cu124 cuda: 12.4
device: NVIDIA GeForce RTX 4090
MAX_JOBS: 32
GRID_SIZE: 512 CHUNK_TRIANGLES: 20000 BOUNDARY_CHUNK_STEPS: 64
weights: 1.0 1.0 0.001
SUBDIVIDE_AREA_THRESHOLD: 3.814697265625e-06 SUBDIVIDE_MAX_ITERS: 12


## JIT extension

The temporary binding includes the CPU implementation by including `flexible_dual_grid.cpp`, and links the three GPU full pipeline versions plus their module dependencies.

In [2]:
build_dir = Path(tempfile.mkdtemp(prefix='fdg_mesh_to_fdg_jit_'))
source_dir = build_dir / 'src'
ext_build_dir = build_dir / 'torch_ext'
source_dir.mkdir(parents=True, exist_ok=True)
ext_build_dir.mkdir(parents=True, exist_ok=True)

binding_cpp = f'''
#include <torch/extension.h>
#include "{(ROOT / 'src/convert/flexible_dual_grid.cpp').as_posix()}"
#include "{(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/api.h').as_posix()}"

std::tuple<torch::Tensor, torch::Tensor, torch::Tensor>
mesh_to_fdg_cpu_wrap(
    const torch::Tensor &vertices,
    const torch::Tensor &faces,
    const torch::Tensor &voxel_size,
    const torch::Tensor &grid_range,
    float face_weight,
    float boundary_weight,
    float regularization_weight) {{
    return mesh_to_flexible_dual_grid_cpu(vertices, faces, voxel_size, grid_range, face_weight, boundary_weight, regularization_weight, false);
}}

std::tuple<torch::Tensor, torch::Tensor, torch::Tensor>
mesh_to_fdg_old_wrap(
    const torch::Tensor &vertices,
    const torch::Tensor &faces,
    const torch::Tensor &voxel_size,
    const torch::Tensor &grid_range,
    float face_weight,
    float boundary_weight,
    float regularization_weight,
    int64_t intersect_chunk_triangles,
    int boundary_chunk_steps) {{
    return o_voxel::fdg::mesh_to_flexible_dual_grid_old(vertices, faces, voxel_size, grid_range, face_weight, boundary_weight, regularization_weight, intersect_chunk_triangles, boundary_chunk_steps);
}}

std::tuple<torch::Tensor, torch::Tensor, torch::Tensor>
mesh_to_fdg_ref_wrap(
    const torch::Tensor &vertices,
    const torch::Tensor &faces,
    const torch::Tensor &voxel_size,
    const torch::Tensor &grid_range,
    float face_weight,
    float boundary_weight,
    float regularization_weight,
    int64_t intersect_chunk_triangles) {{
    return o_voxel::fdg::mesh_to_flexible_dual_grid_ref(vertices, faces, voxel_size, grid_range, face_weight, boundary_weight, regularization_weight, intersect_chunk_triangles);
}}

std::tuple<torch::Tensor, torch::Tensor, torch::Tensor>
mesh_to_fdg_new_wrap(
    const torch::Tensor &vertices,
    const torch::Tensor &faces,
    const torch::Tensor &voxel_size,
    const torch::Tensor &grid_range,
    float face_weight,
    float boundary_weight,
    float regularization_weight,
    int64_t intersect_chunk_triangles) {{
    return o_voxel::fdg::mesh_to_flexible_dual_grid(vertices, faces, voxel_size, grid_range, face_weight, boundary_weight, regularization_weight, intersect_chunk_triangles);
}}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {{
    m.def("mesh_to_fdg_cpu", &mesh_to_fdg_cpu_wrap);
    m.def("mesh_to_fdg_old", &mesh_to_fdg_old_wrap);
    m.def("mesh_to_fdg_ref", &mesh_to_fdg_ref_wrap);
    m.def("mesh_to_fdg_new", &mesh_to_fdg_new_wrap);
}}
'''

binding_path = source_dir / 'binding.cpp'
binding_path.write_text(binding_cpp)

cu_sources = [
    ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/intersect_qef_old.cu',
    ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/intersect_qef_ref.cu',
    ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/intersect_qef.cu',
    ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/voxelize_mesh_octree.cu',
    ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/face_qef_old.cu',
    ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/face_qef_ref.cu',
    ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/face_qef.cu',
    ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/boundary_qef_old.cu',
    ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/boundary_qef_ref.cu',
    ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/boundary_qef.cu',
    ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/mesh_to_flexible_dual_grid_old.cu',
    ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/mesh_to_flexible_dual_grid_ref.cu',
    ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/mesh_to_flexible_dual_grid.cu',
]

ext = load(
    name=f'fdg_mesh_to_fdg_jit_{os.getpid()}',
    sources=[str(binding_path)] + [str(p) for p in cu_sources],
    extra_include_paths=[
        str(ROOT / 'src/convert'),
        str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu'),
        str(ROOT / 'third_party/eigen'),
    ],
    extra_cflags=['-O3', '-std=c++17'],
    extra_cuda_cflags=['-O3', '-std=c++17', '-ccbin', str(CONDA_PREFIX / 'bin/gcc')],
    with_cuda=True,
    verbose=True,
    build_directory=str(ext_build_dir),
)
print('JIT extension loaded:', ext)

Detected CUDA files, patching ldflags
Emitting ninja build file /tmp/fdg_mesh_to_fdg_jit_j5r3g5r_/torch_ext/build.ninja...
/home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module fdg_mesh_to_fdg_jit_1793841...
Using envvar MAX_JOBS (32) as the number of workers...


[1/15] /home/quanta/.conda/envs/symm-enforce/bin/g++ -MMD -MF binding.o.d -DTORCH_EXTENSION_NAME=fdg_mesh_to_fdg_jit_1793841 -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -I/mnt/nvmefs/Projects/o-voxel-gpu/src/convert -I/mnt/nvmefs/Projects/o-voxel-gpu/src/convert/mesh_to_flexible_dual_grid_gpu -I/mnt/nvmefs/Projects/o-voxel-gpu/third_party/eigen -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include/torch/csrc/api/include -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include/TH -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include/THC -isystem /home/quanta/.conda/envs/symm-enforce/include -isystem /home/quanta/.conda/envs/symm-enforce/include/python3.10 -D_GLIBCXX_USE_CXX11_ABI=0 -fPIC -std=c++17 -O3 -s

Loading extension module fdg_mesh_to_fdg_jit_1793841...


## Load original and subdivided meshes

The mesh is normalized into `[0, 1]^3`. The subdivision helper is the same topology-aware PyTorch3D routine used in the earlier intersect notebook.

In [3]:
def subdivide_mesh_gpu(verts: torch.Tensor, faces: torch.Tensor, area_threshold: float, max_iters: int):
    from pytorch3d.structures import Meshes

    device = verts.device
    cur = Meshes(verts=[verts], faces=[faces])

    for _ in range(max_iters):
        cv = cur.verts_packed()
        cf = cur.faces_packed()
        edges = cur.edges_packed()
        f2e = cur.faces_packed_to_edges_packed()

        areas = cur.faces_areas_packed()
        large = areas > area_threshold
        if not large.any():
            break

        edge_len = torch.norm(cv[edges[:, 0]] - cv[edges[:, 1]], dim=1)
        _, local_max = torch.max(edge_len[f2e], dim=1)
        max_edge_ids = f2e[torch.arange(cf.shape[0], device=device), local_max]

        target = torch.zeros(edges.shape[0], dtype=torch.bool, device=device)
        target[max_edge_ids[large]] = True
        if not target.any():
            break

        mid = (cv[edges[target, 0]] + cv[edges[target, 1]]) * 0.5
        new_verts = torch.cat([cv, mid], dim=0)
        n_old = cv.shape[0]

        e2new = torch.zeros(edges.shape[0], dtype=torch.long, device=device)
        e2new[target] = torch.arange(n_old, n_old + target.sum().item(), device=device)

        split_counts = target[f2e].sum(dim=1)
        v0, v1, v2 = cf[:, 0], cf[:, 1], cf[:, 2]
        p01_a = torch.minimum(v0, v1); p01_b = torch.maximum(v0, v1)
        p12_a = torch.minimum(v1, v2); p12_b = torch.maximum(v1, v2)
        p20_a = torch.minimum(v2, v0); p20_b = torch.maximum(v2, v0)

        e_act = edges[f2e]
        match01 = (e_act[:, :, 0] == p01_a.unsqueeze(1)) & (e_act[:, :, 1] == p01_b.unsqueeze(1))
        match12 = (e_act[:, :, 0] == p12_a.unsqueeze(1)) & (e_act[:, :, 1] == p12_b.unsqueeze(1))
        match20 = (e_act[:, :, 0] == p20_a.unsqueeze(1)) & (e_act[:, :, 1] == p20_b.unsqueeze(1))
        col01 = match01.long().argmax(dim=1)
        col12 = match12.long().argmax(dim=1)
        col20 = match20.long().argmax(dim=1)

        keep = cf[split_counts == 0]
        out = [keep]

        m1 = split_counts == 1
        if m1.any():
            f1 = cf[m1]; fe1 = f2e[m1]; M = f1.shape[0]
            v0_1, v1_1, v2_1 = f1[:, 0], f1[:, 1], f1[:, 2]
            split_col = target[fe1].long().argmax(dim=1)
            is_split_01 = split_col == col01[m1]
            is_split_12 = split_col == col12[m1]
            is_split_20 = split_col == col20[m1]
            split_eid = fe1[torch.arange(M, device=device), split_col]
            vn = e2new[split_eid]
            s1 = torch.zeros(M, 3, dtype=torch.long, device=device)
            s2 = torch.zeros(M, 3, dtype=torch.long, device=device)
            s1[is_split_01] = torch.stack([v0_1[is_split_01], vn[is_split_01], v2_1[is_split_01]], dim=1)
            s2[is_split_01] = torch.stack([vn[is_split_01], v1_1[is_split_01], v2_1[is_split_01]], dim=1)
            s1[is_split_12] = torch.stack([v0_1[is_split_12], v1_1[is_split_12], vn[is_split_12]], dim=1)
            s2[is_split_12] = torch.stack([v0_1[is_split_12], vn[is_split_12], v2_1[is_split_12]], dim=1)
            s1[is_split_20] = torch.stack([v0_1[is_split_20], v1_1[is_split_20], vn[is_split_20]], dim=1)
            s2[is_split_20] = torch.stack([vn[is_split_20], v1_1[is_split_20], v2_1[is_split_20]], dim=1)
            out.extend([s1, s2])

        mm = split_counts >= 2
        if mm.any():
            fm = cf[mm]; fem = f2e[mm]; M2 = fm.shape[0]
            v0_m, v1_m, v2_m = fm[:, 0], fm[:, 1], fm[:, 2]
            eid01 = fem[torch.arange(M2, device=device), col01[mm]]
            eid12 = fem[torch.arange(M2, device=device), col12[mm]]
            eid20 = fem[torch.arange(M2, device=device), col20[mm]]
            e01_v = torch.where(target[eid01], e2new[eid01], v0_m)
            e12_v = torch.where(target[eid12], e2new[eid12], v1_m)
            e20_v = torch.where(target[eid20], e2new[eid20], v2_m)
            out.extend([
                torch.stack([v0_m, e01_v, e20_v], dim=1),
                torch.stack([e01_v, v1_m, e12_v], dim=1),
                torch.stack([e20_v, e12_v, v2_m], dim=1),
                torch.stack([e01_v, e12_v, e20_v], dim=1),
            ])

        cur = Meshes(verts=[new_verts], faces=[torch.cat(out, dim=0)])

    return cur.verts_packed(), cur.faces_packed()


def tensor_mb(x: torch.Tensor) -> float:
    return x.numel() * x.element_size() / 1024**2


glb_path = ROOT / 'notebooks/test.glb'
loaded = trimesh.load(glb_path, force='scene')
if isinstance(loaded, trimesh.Scene):
    meshes = [g for g in loaded.geometry.values() if isinstance(g, trimesh.Trimesh) and len(g.faces) > 0]
    mesh = trimesh.util.concatenate(meshes)
else:
    mesh = loaded
assert isinstance(mesh, trimesh.Trimesh) and len(mesh.faces) > 0

vertices_np = np.asarray(mesh.vertices, dtype=np.float32)
faces_np = np.asarray(mesh.faces, dtype=np.int64)
vmin = vertices_np.min(axis=0); vmax = vertices_np.max(axis=0)
center = (vmin + vmax) * 0.5
scale = np.float32(0.999 / max((vmax - vmin).max(), 1e-12))
vertices_np = (vertices_np - center) * scale + np.float32(0.5)

vertices_cpu = torch.from_numpy(vertices_np).contiguous()
faces_cpu = torch.from_numpy(faces_np).to(torch.int32).contiguous()
vertices_cuda = vertices_cpu.to(DEVICE, non_blocking=True).contiguous()
faces_cuda = faces_cpu.to(DEVICE, non_blocking=True).contiguous()
voxel_size_cpu = torch.tensor([1.0 / GRID_SIZE, 1.0 / GRID_SIZE, 1.0 / GRID_SIZE], dtype=torch.float32)
grid_range_cpu = torch.tensor([0, 0, 0, GRID_SIZE, GRID_SIZE, GRID_SIZE], dtype=torch.int32)

print('building subdivided input...')
torch.cuda.empty_cache(); torch.cuda.synchronize()
start = time.perf_counter()
sub_vertices_cuda, sub_faces_cuda = subdivide_mesh_gpu(vertices_cuda, faces_cuda.to(torch.long), SUBDIVIDE_AREA_THRESHOLD, SUBDIVIDE_MAX_ITERS)
sub_faces_cuda = sub_faces_cuda.to(torch.int32).contiguous()
torch.cuda.synchronize()
subdivide_ms = (time.perf_counter() - start) * 1000.0
sub_vertices_cpu = sub_vertices_cuda.detach().cpu().contiguous()
sub_faces_cpu = sub_faces_cuda.detach().cpu().contiguous()
torch.cuda.empty_cache()

INPUT_CASES = OrderedDict([
    ('original', dict(vertices_cpu=vertices_cpu, faces_cpu=faces_cpu, vertices_cuda=vertices_cuda, faces_cuda=faces_cuda,
                      vertices_shape=tuple(vertices_cpu.shape), faces_shape=tuple(faces_cpu.shape),
                      vertex_mb=tensor_mb(vertices_cuda), face_mb=tensor_mb(faces_cuda), preprocess_ms=0.0)),
    ('subdivided', dict(vertices_cpu=sub_vertices_cpu, faces_cpu=sub_faces_cpu, vertices_cuda=sub_vertices_cuda, faces_cuda=sub_faces_cuda,
                       vertices_shape=tuple(sub_vertices_cpu.shape), faces_shape=tuple(sub_faces_cpu.shape),
                       vertex_mb=tensor_mb(sub_vertices_cuda), face_mb=tensor_mb(sub_faces_cuda), preprocess_ms=subdivide_ms)),
])
for name, case in INPUT_CASES.items():
    print(name, 'vertices:', case['vertices_shape'], 'faces:', case['faces_shape'], 'vertex_mb:', case['vertex_mb'], 'face_mb:', case['face_mb'], 'preprocess_ms:', case['preprocess_ms'])
print('face multiplier:', INPUT_CASES['subdivided']['faces_shape'][0] / max(INPUT_CASES['original']['faces_shape'][0], 1))
print('voxel_size:', voxel_size_cpu.tolist(), 'grid_range:', grid_range_cpu.tolist())

building subdivided input...
original vertices: (268018, 3) faces: (280333, 3) vertex_mb: 3.0672225952148438 face_mb: 3.2081565856933594 preprocess_ms: 0.0
subdivided vertices: (3608864, 3) faces: (10304077, 3) vertex_mb: 41.3001708984375 face_mb: 117.92080307006836 preprocess_ms: 803.9257089985767
face multiplier: 36.75656094715927
voxel_size: [0.001953125, 0.001953125, 0.001953125] grid_range: [0, 0, 0, 512, 512, 512]


## Canonicalization and wrappers

Outputs are sorted by packed voxel key before comparison, so different output orders do not affect correctness metrics.

In [4]:
def mesh_to_fdg_cpu_py(case):
    return ext.mesh_to_fdg_cpu(case['vertices_cpu'], case['faces_cpu'], voxel_size_cpu, grid_range_cpu, FACE_WEIGHT, BOUNDARY_WEIGHT, REGULARIZATION_WEIGHT)


def mesh_to_fdg_old_py(case):
    return ext.mesh_to_fdg_old(case['vertices_cuda'], case['faces_cuda'], voxel_size_cpu, grid_range_cpu, FACE_WEIGHT, BOUNDARY_WEIGHT, REGULARIZATION_WEIGHT, CHUNK_TRIANGLES, BOUNDARY_CHUNK_STEPS)


def mesh_to_fdg_ref_py(case):
    return ext.mesh_to_fdg_ref(case['vertices_cuda'], case['faces_cuda'], voxel_size_cpu, grid_range_cpu, FACE_WEIGHT, BOUNDARY_WEIGHT, REGULARIZATION_WEIGHT, CHUNK_TRIANGLES)


def mesh_to_fdg_new_py(case):
    return ext.mesh_to_fdg_new(case['vertices_cuda'], case['faces_cuda'], voxel_size_cpu, grid_range_cpu, FACE_WEIGHT, BOUNDARY_WEIGHT, REGULARIZATION_WEIGHT, CHUNK_TRIANGLES)


def voxel_keys(voxels: torch.Tensor) -> torch.Tensor:
    voxels = voxels.to(torch.int64)
    rel = voxels - grid_range_cpu[:3].to(torch.int64)
    size = (grid_range_cpu[3:] - grid_range_cpu[:3]).to(torch.int64)
    return rel[:, 0] + size[0] * (rel[:, 1] + size[1] * rel[:, 2])


def canonicalize_output(output):
    voxels, dual_vertices, intersected = output
    voxels = voxels.detach().cpu().to(torch.int64).contiguous()
    dual_vertices = dual_vertices.detach().cpu().float().contiguous()
    intersected = intersected.detach().cpu().bool().contiguous()
    keys = voxel_keys(voxels)
    order = torch.argsort(keys)
    keys = keys[order]
    return {'keys': keys, 'voxels': voxels[order], 'dual_vertices': dual_vertices[order], 'intersected': intersected[order], 'unique': bool(keys.numel() == 0 or torch.all(keys[1:] != keys[:-1]).item())}


def common_sorted_indices(a: torch.Tensor, b: torch.Tensor):
    if a.numel() == 0 or b.numel() == 0:
        return torch.empty(0, dtype=torch.long), torch.empty(0, dtype=torch.long)
    pos = torch.searchsorted(b, a)
    valid = pos < b.numel()
    eq = torch.zeros_like(valid, dtype=torch.bool)
    eq[valid] = b[pos[valid]] == a[valid]
    ia = torch.nonzero(eq, as_tuple=False).flatten()
    return ia, pos[ia]


def compare_outputs(cpu, other):
    ia, ib = common_sorted_indices(cpu['keys'], other['keys'])
    occ_equal = bool(cpu['keys'].numel() == other['keys'].numel() and ia.numel() == cpu['keys'].numel())
    row = {'occ_equal': occ_equal, 'cpu_count': int(cpu['keys'].numel()), 'other_count': int(other['keys'].numel()), 'common_count': int(ia.numel()), 'missing': int(cpu['keys'].numel() - ia.numel()), 'extra': int(other['keys'].numel() - ib.numel()), 'other_unique': other['unique']}
    if ia.numel() == 0:
        row.update({'intersected_mismatch': None, 'dual_max_abs': None, 'dual_mean_abs': None, 'dual_rms': None, 'dual_p99_abs': None, 'dual_max_abs_voxels': None})
        return row
    dual_diff = other['dual_vertices'][ib] - cpu['dual_vertices'][ia]
    abs_diff = dual_diff.abs()
    row.update({
        'intersected_mismatch': int((other['intersected'][ib] != cpu['intersected'][ia]).sum().item()),
        'dual_max_abs': float(abs_diff.max().item()),
        'dual_mean_abs': float(abs_diff.mean().item()),
        'dual_rms': float(torch.sqrt((dual_diff * dual_diff).mean()).item()),
        'dual_p99_abs': float(torch.quantile(abs_diff.flatten(), 0.99).item()),
        'dual_max_abs_voxels': float(abs_diff.max().item() / voxel_size_cpu.max().item()),
    })
    return row

## Correctness against CPU

Both original and subdivided meshes run the full CPU pipeline. GPU outputs are compared to CPU after voxel-key canonicalization.

In [5]:
CASE_CANON = {}
correctness_rows = []
DUAL_MAX_ABS_LIMIT = 2.0 * float(voxel_size_cpu.max().item())

for input_name, case in INPUT_CASES.items():
    print(f'\n=== correctness: {input_name} ===')
    outputs = OrderedDict()
    outputs['cpu'] = mesh_to_fdg_cpu_py(case)
    outputs['old'] = mesh_to_fdg_old_py(case)
    outputs['ref'] = mesh_to_fdg_ref_py(case)
    outputs['new'] = mesh_to_fdg_new_py(case)
    torch.cuda.synchronize()
    canon = {name: canonicalize_output(out) for name, out in outputs.items()}
    CASE_CANON[input_name] = canon
    assert canon['cpu']['unique'], f'{input_name} CPU produced duplicate voxel keys'
    for method in ['old', 'ref', 'new']:
        row = {'input': input_name, 'method': method, **compare_outputs(canon['cpu'], canon[method])}
        correctness_rows.append(row)
        assert row['occ_equal'], f'{input_name} {method} occupancy differs from CPU'
        assert row['intersected_mismatch'] == 0, f'{input_name} {method} intersected flags differ from CPU'
        assert row['dual_max_abs'] <= DUAL_MAX_ABS_LIMIT, f'{input_name} {method} dual max error {row["dual_max_abs"]} exceeds {DUAL_MAX_ABS_LIMIT}'

correctness_df = pd.DataFrame(correctness_rows)
display(correctness_df)
print('dual max abs limit:', DUAL_MAX_ABS_LIMIT, 'voxel_size:', float(voxel_size_cpu.max().item()))


=== correctness: original ===

=== correctness: subdivided ===


,input,method,occ_equal,cpu_count,other_count,common_count,missing,extra,other_unique,intersected_mismatch,dual_max_abs,dual_mean_abs,dual_rms,dual_p99_abs,dual_max_abs_voxels
0,original,old,True,4676307,4676307,4676307,0,0,True,0,0.001953,0.000020,0.000091,0.000191,1.0
1,original,ref,True,4676307,4676307,4676307,0,0,True,0,0.001953,0.000051,0.000166,0.000966,1.0
2,original,new,True,4676307,4676307,4676307,0,0,True,0,0.001953,0.000020,0.000091,0.000191,1.0
3,subdivided,old,True,4676306,4676306,4676306,0,0,True,0,0.001953,0.000036,0.000116,0.000358,1.0
4,subdivided,ref,True,4676306,4676306,4676306,0,0,True,0,0.001953,0.000078,0.000208,0.000999,1.0
5,subdivided,new,True,4676306,4676306,4676306,0,0,True,0,0.001953,0.000037,0.000116,0.000359,1.0


dual max abs limit: 0.00390625 voxel_size: 0.001953125


## GPU timing and PyTorch-visible memory

Cold timing clears the PyTorch cache before one call. Warm timing performs one unmeasured warmup and reports median/min over repeated CUDA-event timings.

In [6]:
def timed_cuda_call(fn, cold=False):
    if cold:
        torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    base_alloc = torch.cuda.memory_allocated()
    base_reserved = torch.cuda.memory_reserved()
    start = torch.cuda.Event(enable_timing=True); end = torch.cuda.Event(enable_timing=True)
    start.record(); out = fn(); end.record(); torch.cuda.synchronize()
    ms = start.elapsed_time(end)
    peak_alloc = torch.cuda.max_memory_allocated() - base_alloc
    peak_reserved = torch.cuda.max_memory_reserved() - base_reserved
    del out
    torch.cuda.synchronize()
    return {'ms': float(ms), 'peak_alloc_mb': peak_alloc / 1024**2, 'peak_reserved_mb': peak_reserved / 1024**2}


def summarize_records(records):
    ms = np.array([r['ms'] for r in records], dtype=np.float64)
    alloc = np.array([r['peak_alloc_mb'] for r in records], dtype=np.float64)
    reserved = np.array([r['peak_reserved_mb'] for r in records], dtype=np.float64)
    return {'warm_ms_median': float(np.median(ms)), 'warm_ms_min': float(ms.min()), 'warm_peak_alloc_mb_median': float(np.median(alloc)), 'warm_peak_reserved_mb_median': float(np.median(reserved))}


timing_rows = []
for input_name, case in INPUT_CASES.items():
    print(f'\n=== timing: {input_name} ===')
    methods = OrderedDict([
        ('old', lambda case=case: mesh_to_fdg_old_py(case)),
        ('ref', lambda case=case: mesh_to_fdg_ref_py(case)),
        ('new', lambda case=case: mesh_to_fdg_new_py(case)),
    ])
    for method, fn in methods.items():
        print('timing', input_name, method)
        cold = timed_cuda_call(fn, cold=True)
        _ = timed_cuda_call(fn, cold=False)
        warm_records = [timed_cuda_call(fn, cold=False) for _ in range(WARM_REPEATS)]
        timing_rows.append({'input': input_name, 'method': method, 'vertices': case['vertices_shape'][0], 'faces': case['faces_shape'][0], 'input_vertex_mb': case['vertex_mb'], 'input_face_mb': case['face_mb'], 'cold_ms': cold['ms'], 'cold_peak_alloc_mb': cold['peak_alloc_mb'], 'cold_peak_reserved_mb': cold['peak_reserved_mb'], **summarize_records(warm_records)})

timing_df = pd.DataFrame(timing_rows)
display(timing_df)


=== timing: original ===
timing original old
timing original ref
timing original new

=== timing: subdivided ===
timing subdivided old
timing subdivided ref
timing subdivided new


,input,method,vertices,faces,input_vertex_mb,input_face_mb,cold_ms,cold_peak_alloc_mb,cold_peak_reserved_mb,warm_ms_median,warm_ms_min,warm_peak_alloc_mb_median,warm_peak_reserved_mb_median
0,original,old,268018,280333,3.067223,3.208157,152.465958,2980.857422,580.0,140.725250,138.002426,2980.857422,0.0
1,original,ref,268018,280333,3.067223,3.208157,142.077026,1968.839844,2.0,139.358215,137.499649,1968.839844,0.0
2,original,new,268018,280333,3.067223,3.208157,26.916864,939.653809,2.0,27.063616,26.562559,939.653809,0.0
3,subdivided,old,3608864,10304077,41.300171,117.920803,598.421692,17746.291992,19810.0,548.899475,521.495850,17746.291992,0.0
4,subdivided,ref,3608864,10304077,41.300171,117.920803,113.937408,12959.359863,10242.0,102.189278,101.845695,12959.359863,0.0
5,subdivided,new,3608864,10304077,41.300171,117.920803,85.948418,1836.847168,2.0,82.050049,81.422333,1836.847168,0.0


## Compact summary

In [7]:
summary = correctness_df.merge(timing_df, on=['input', 'method'], how='left')
display(summary[[
    'input', 'method', 'occ_equal', 'cpu_count', 'other_count', 'intersected_mismatch',
    'dual_max_abs', 'dual_mean_abs', 'dual_rms', 'dual_max_abs_voxels',
    'cold_ms', 'warm_ms_median', 'warm_ms_min', 'cold_peak_alloc_mb', 'warm_peak_alloc_mb_median',
]])

,input,method,occ_equal,cpu_count,other_count,intersected_mismatch,dual_max_abs,dual_mean_abs,dual_rms,dual_max_abs_voxels,cold_ms,warm_ms_median,warm_ms_min,cold_peak_alloc_mb,warm_peak_alloc_mb_median
0,original,old,True,4676307,4676307,0,0.001953,0.000020,0.000091,1.0,152.465958,140.725250,138.002426,2980.857422,2980.857422
1,original,ref,True,4676307,4676307,0,0.001953,0.000051,0.000166,1.0,142.077026,139.358215,137.499649,1968.839844,1968.839844
2,original,new,True,4676307,4676307,0,0.001953,0.000020,0.000091,1.0,26.916864,27.063616,26.562559,939.653809,939.653809
3,subdivided,old,True,4676306,4676306,0,0.001953,0.000036,0.000116,1.0,598.421692,548.899475,521.495850,17746.291992,17746.291992
4,subdivided,ref,True,4676306,4676306,0,0.001953,0.000078,0.000208,1.0,113.937408,102.189278,101.845695,12959.359863,12959.359863
5,subdivided,new,True,4676306,4676306,0,0.001953,0.000037,0.000116,1.0,85.948418,82.050049,81.422333,1836.847168,1836.847168
